# DoRA Image/Video-Text Understanding

## 1. Environment Setup

In [ ]:
!pip install -q transformers datasets h5py matplotlib pandas tqdm

import os
import gc
import math
import h5py
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm


print("Environment Setup Complete")
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA is available! Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Using CPU. Training will be extremely slow.")


## 2. Downloading Pre-extracted Image Features

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

In [ ]:
import os
import json

# Define paths
DRIVE_DIR = "/content/drive/My Drive/Colab_Notebooks/DoRA/IVT Understanding/VLT5_Data"
LOCAL_DIR = "/content"

# 1. Switch to Colab's local free disk
%cd {LOCAL_DIR}

print("\n1. Copying execution scripts from Google Drive to local...")
!cp "{DRIVE_DIR}/tsv_to_h5.py" .
!cp "{DRIVE_DIR}/coco_val_compact.py" .

print("\n2. Reading raw data directly from Drive and merging to local...")
def merge_vqa_data(q_path, a_path, output_name):
    # Read directly across directories to save time copying large raw JSONs
    with open(f"{DRIVE_DIR}/{q_path}", 'r') as f: questions = json.load(f)['questions']
    with open(f"{DRIVE_DIR}/{a_path}", 'r') as f: annotations = json.load(f)['annotations']
    ans_map = {ann['question_id']: ann['multiple_choice_answer'] for ann in annotations}
    merged = [{"img_id": str(q['image_id']), "question": q['question'], "answer": ans_map[q['question_id']]} for q in questions if q['question_id'] in ans_map]

    # Write results to /content/
    with open(output_name, 'w') as f: json.dump(merged, f)
    print(f"✅ Created {output_name} with {len(merged)} samples.")


merge_vqa_data(
    'v2_OpenEnded_mscoco_train2014_questions.json',
    'v2_mscoco_train2014_annotations.json',
    'vqa_train.json'
)
merge_vqa_data(
    'v2_OpenEnded_mscoco_val2014_questions.json',
    'v2_mscoco_val2014_annotations.json',
    'vqa_valid.json'
)

print("\n3. Processing Validation Image Features")
# Use -c for resuming downloads if interrupted
!wget -c https://nlp.cs.unc.edu/data/lxmert_data/mscoco_imgfeat/val2014_obj36.zip
!unzip -o val2014_obj36.zip
!python tsv_to_h5.py --tsv_path val2014_obj36.tsv --h5_path val2014_obj36.h5
print(">> Validation conversion complete. Deleting intermediate files to free up space immediately!")
!rm val2014_obj36.zip val2014_obj36.tsv

print("\n4. Extracting Compact Validation Set")
!python coco_val_compact.py --source_h5 val2014_obj36.h5 --target_h5 resplit_val_obj36.h5 --json_ref vqa_valid.json
print(">> Extraction complete. Deleting the massive original validation H5!")
!rm val2014_obj36.h5

print("\n5. Processing Training Image Features")
!wget -c https://nlp.cs.unc.edu/data/lxmert_data/mscoco_imgfeat/train2014_obj36.zip
!unzip -o train2014_obj36.zip
!python tsv_to_h5.py --tsv_path train2014_obj36.tsv --h5_path train2014_obj36.h5
print(">> Training conversion complete. Deleting intermediate files to free up space immediately!")
!rm train2014_obj36.zip train2014_obj36.tsv

print("\n✅ All data is now ready in the Colab local directory (/content) without using Google Drive storage!")

## 3. Configuration


In [ ]:
class Config:
    # Model Architecture Config
    MODEL_NAME = "t5-base"        # HuggingFace backbone
    HIDDEN_SIZE = 768             # T5-base hidden dimension
    VISUAL_DIM = 2048             # Faster R-CNN feature dimension
    BOX_DIM = 4                   # Bounding box coordinates dimension
    MAX_TEXT_LEN = 20             # Max length for input prompt
    MAX_ANS_LEN = 10              # Max length for generated answer
    MAX_OBJECTS = 36              # Number of object proposals per image

    # Tuning Strategy Config (DoRA / LoRA)
    RANK = 16
    ALPHA = 32
    DROPOUT = 0.1

    # Training Config
    BATCH_SIZE = 4                # Keep small for testing; increase based on VRAM
    EPOCHS = 3
    LR = 1e-4

    # Hardware Config
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Using bfloat16 if supported for memory efficiency, else float32
    DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

cfg = Config()

print("✅ Configuration Loaded")
print(f"Backbone: {cfg.MODEL_NAME} | Hidden Size: {cfg.HIDDEN_SIZE}")
print(f"DoRA Rank: {cfg.RANK} | Alpha: {cfg.ALPHA}")
print(f"Device: {cfg.DEVICE} | Compute Dtype: {cfg.DTYPE}")
print(f"Configuration loaded.")

## 3. Fine-tuning Layer: DoRA, LoRA, Full

For a frozen pretrained weight `W0`, DoRA trains a magnitude vector `m` and low-rank direction update `BA`:

$$W' = m \frac{W_0 + BA}{\lVert W_0 + BA \rVert_c}$$

PyTorch `Linear` stores weights as `(out_features, in_features)`, so this implementation normalizes each output row. This is equivalent to column-wise normalization under the transposed convention used by `F.linear`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = F.linear(x, self.weight, self.bias)
        if self.training:
            lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling
        else:
            lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scaling
        return base_out + lora_out

class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Freeze Original Weights
        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        # LoRA Parameters
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # Magnitude Parameters
        magnitude = torch.linalg.vector_norm(self.weight.float(), ord=2, dim=1).to(self.weight.dtype)
        self.magnitude = nn.Parameter(magnitude)

    def merged_weight(self):
        """Merge weight for inference time"""
        directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1, keepdim=True).to(directional.dtype).clamp_min(1e-6)
        return self.magnitude[:, None] * directional / norm

    def forward(self, x):
        if self.training:
            directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
            norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1).to(directional.dtype).clamp_min(1e-6)

            # note that authors mentioned norm.detach() here as a trick to reduce training cost
            # here we removed .detach() to enable full orthogonal gradient projection
            norm_scale = self.magnitude / norm

            base_out = F.linear(x, self.weight, None)
            lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling

            out = norm_scale * (base_out + lora_out)

            if self.bias is not None:
                out = out + self.bias

            return out

        else:
            return F.linear(x, self.merged_weight(), self.bias)


def inject_and_unfreeze(model: nn.Module, target_modules: list, rank: int, alpha: int, dropout: float, method: str) -> tuple:
    """return (# of substituted layer, # of trainable params, # of total params)"""
    model.requires_grad_(False)

    modules = dict(model.named_modules())
    replacements = []


    for name, module in modules.items():
        if isinstance(module, nn.Linear) and any(target in name for target in target_modules):
            replacements.append(name)

    replace_count = 0
    for name in replacements:
        parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
        parent = modules[parent_name] if parent_name else model
        old_layer = getattr(parent, child_name)

        if method == "dora":
            new_layer = DoRALinear(old_layer, rank=rank, alpha=alpha, dropout=dropout)
            setattr(parent, child_name, new_layer)
            # Unfreeze DoRA parameters
            new_layer.lora_A.requires_grad_(True)
            new_layer.lora_B.requires_grad_(True)
            new_layer.magnitude.requires_grad_(True)
            replace_count += 1

        elif method == "lora":
            new_layer = LoRALinear(old_layer, rank=rank, alpha=alpha, dropout=dropout)
            setattr(parent, child_name, new_layer)
            # Unfreeze LoRA parameters
            new_layer.lora_A.requires_grad_(True)
            new_layer.lora_B.requires_grad_(True)
            replace_count += 1

        elif method == "full":
            # Unfreeze complete target layer
            old_layer.weight.requires_grad_(True)
            if old_layer.bias is not None:
                old_layer.bias.requires_grad_(True)
            replace_count += 1

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return replace_count, trainable, total

print("✅ Core modules defined.")

### Smoketest Sanity Check For DoRA

In [ ]:
print("Running DoRALinear startup identity test...")

_test_layer = nn.Linear(7, 4, bias=False)
_wrap_layer = DoRALinear(_test_layer, rank=2, alpha=4, dropout=0.0)
_dummy_input = torch.randn(3, 7)

# Verify outputs match exactly at initialization
with torch.no_grad():
    assert torch.allclose(_test_layer(_dummy_input), _wrap_layer(_dummy_input), atol=1e-5), "Mismatch detected!"

print("SUCCESS: Identity test passed. DoRA starts cleanly.")

In [ ]:
print("--- Running DoRA Injection Smoketest ---")

# 1. Create a dummy model
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

dummy_model = DummyModel().to(cfg.DEVICE)
original_total_params = sum(p.numel() for p in dummy_model.parameters())

# 2. Inject DoRA into 'fc1' and 'fc2'
target_modules = ["fc1", "fc2"]
replaced, trainable_params, total_params = inject_and_unfreeze(
    dummy_model,
    target_modules,
    rank=cfg.RANK,
    alpha=cfg.ALPHA,
    dropout=cfg.DROPOUT,
    method="dora"
)

print(f"Layers Replaced: {replaced}")
print(f"Trainable Params: {trainable_params} / {total_params} ({(trainable_params/total_params)*100:.2f}%)")

assert replaced == 2, "Failed to replace the correct number of layers!"

# 3. Test Forward and Backward pass
dummy_model.train()
dummy_input = torch.randn(8, 128).to(cfg.DEVICE)
dummy_target = torch.randn(8, 10).to(cfg.DEVICE)

output = dummy_model(dummy_input)
loss = F.mse_loss(output, dummy_target)
loss.backward()

# 4. Verify Gradients (CRITICAL CHECK)
# Check fc1 (which is now a DoRALinear)
fc1_layer = dummy_model.fc1
assert fc1_layer.lora_A.grad is not None, "Error: lora_A did not receive gradients!"
assert fc1_layer.lora_B.grad is not None, "Error: lora_B did not receive gradients!"
assert fc1_layer.magnitude.grad is not None, "Error: magnitude did not receive gradients!"
assert fc1_layer.weight.grad is None, "FATAL ERROR: Frozen base weight unexpectedly received gradients!"

print("SUCCESS: Forward and backward passes executed correctly.")
print("SUCCESS: Gradients are flowing ONLY to DoRA parameters. Base weights are frozen.")
print("--- Smoketest Passed! ---")

# Clean up memory
del dummy_model, dummy_input, dummy_target, output, loss
torch.cuda.empty_cache()
gc.collect()

## 4. Core Multi-modal T5 Architecture



In [ ]:
class VisualEmbedding(nn.Module):
    def __init__(self, hidden_size, visual_dim, box_dim):
        super().__init__()
        # Project 2048-d visual features to T5 hidden dimension
        self.vis_projection = nn.Linear(visual_dim, hidden_size)
        # Project 4-d spatial bounding boxes to T5 hidden dimension
        self.box_projection = nn.Linear(box_dim, hidden_size)
        # LayerNorm to stabilize multi-modal fusion
        self.layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, visual_feats, visual_boxes):
        """
        visual_feats: [batch_size, num_objects, visual_dim]
        visual_boxes: [batch_size, num_objects, box_dim]
        """
        vis_embeds = self.vis_projection(visual_feats)
        box_embeds = self.box_projection(visual_boxes)

        # Combine appearance and spatial geometry
        combined_embeds = vis_embeds + box_embeds
        return self.layer_norm(combined_embeds)


class CustomVLT5(nn.Module):
    def __init__(self, config=cfg):
        super().__init__()
        print(f"Initializing Custom Multi-modal T5 based on {config.MODEL_NAME}...")

        # Load standard text-only T5
        self.t5 = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME)
        self.hidden_size = self.t5.config.d_model

        # Add visual embedding layer
        self.visual_embedding = VisualEmbedding(
            hidden_size=self.hidden_size,
            visual_dim=config.VISUAL_DIM,
            box_dim=config.BOX_DIM
        )

    def forward(self, input_ids, attention_mask, visual_feats, visual_boxes, labels=None):
        # 1. Get Text Embeddings from T5's internal word embeddings
        text_embeds = self.t5.shared(input_ids)

        # 2. Get Visual Embeddings
        vis_embeds = self.visual_embedding(visual_feats, visual_boxes)

        # 3. Concatenate Vision and Text along the sequence length dimension (dim=1)
        # Shape: [batch_size, num_objects + seq_len, hidden_size]
        inputs_embeds = torch.cat([vis_embeds, text_embeds], dim=1)

        # 4. Create extended attention mask (Visual features are fully attended to)
        batch_size, num_objects = visual_feats.shape[:2]
        vis_mask = torch.ones((batch_size, num_objects), device=input_ids.device, dtype=attention_mask.dtype)
        combined_attention_mask = torch.cat([vis_mask, attention_mask], dim=1)

        # 5. Forward through T5 body
        return self.t5(
            inputs_embeds=inputs_embeds,
            attention_mask=combined_attention_mask,
            labels=labels
        )

    def generate(self, input_ids, attention_mask, visual_feats, visual_boxes, **kwargs):
        """Autoregressive generation for inference/evaluation."""
        text_embeds = self.t5.shared(input_ids)
        vis_embeds = self.visual_embedding(visual_feats, visual_boxes)
        inputs_embeds = torch.cat([vis_embeds, text_embeds], dim=1)

        batch_size, num_objects = visual_feats.shape[:2]
        vis_mask = torch.ones((batch_size, num_objects), device=input_ids.device, dtype=attention_mask.dtype)
        combined_attention_mask = torch.cat([vis_mask, attention_mask], dim=1)

        return self.t5.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=combined_attention_mask,
            **kwargs
        )

print("✅ Multi-modal Architecture Defined")

### 4.1 Smoketest for Model Architecture

In [ ]:
print("--- Running Architecture Smoketest ---")
model_smoketest = CustomVLT5(cfg).to(cfg.DEVICE)
model_smoketest.train()

# Create dummy inputs mimicking VQA data
bs = 2
dummy_vis_feats = torch.randn(bs, cfg.MAX_OBJECTS, cfg.VISUAL_DIM).to(cfg.DEVICE)
dummy_vis_boxes = torch.randn(bs, cfg.MAX_OBJECTS, cfg.BOX_DIM).to(cfg.DEVICE)
dummy_input_ids = torch.randint(0, 30000, (bs, cfg.MAX_TEXT_LEN)).to(cfg.DEVICE)
dummy_attn_mask = torch.ones(bs, cfg.MAX_TEXT_LEN).to(cfg.DEVICE)
dummy_labels = torch.randint(0, 30000, (bs, cfg.MAX_ANS_LEN)).to(cfg.DEVICE)

# Forward pass
outputs = model_smoketest(
    input_ids=dummy_input_ids,
    attention_mask=dummy_attn_mask,
    visual_feats=dummy_vis_feats,
    visual_boxes=dummy_vis_boxes,
    labels=dummy_labels
)

print(f"Output Loss: {outputs.loss.item():.4f}")
print(f"Output Logits Shape: {outputs.logits.shape}")

assert outputs.loss is not None, "Error: Forward pass failed to compute loss."
print("SUCCESS: Architecture forward pass handles multi-modal inputs correctly.")
print("--- Smoketest Passed! ---")

del model_smoketest, outputs
torch.cuda.empty_cache()
gc.collect()

## 5. Custom VQA Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import os
import json
import h5py
import torch

class CustomVQADataset(Dataset):
    def __init__(self, h5_path, qa_json_path, tokenizer, config=cfg, is_train=True):
        self.tokenizer = tokenizer
        self.config = config
        self.is_train = is_train

        if not (os.path.exists(h5_path) and os.path.exists(qa_json_path)):
            raise FileNotFoundError(f"❌ Cannot find real data! Please check paths:\n- H5: {h5_path}\n- JSON: {qa_json_path}")

        print(f"Loading features from {h5_path} and QA from {qa_json_path}...")
        self.features_db = h5py.File(h5_path, 'r')
        with open(qa_json_path, 'r') as f:
            self.qa_data = json.load(f)

        self.data_len = len(self.qa_data)
        print(f"Loaded {self.data_len} samples.")

    def __len__(self):
        return self.data_len

    def __getitem__(self, idx):
        item = self.qa_data[idx]
        raw_img_id = int(item['img_id'])
        question_text = f"vqa: {item['question']}"
        answer_text = item['answer']


        train_key = f"COCO_train2014_{raw_img_id:012d}"
        val_key = f"COCO_val2014_{raw_img_id:012d}"

        if train_key in self.features_db:
            h5_key = train_key
        elif val_key in self.features_db:
            h5_key = val_key
        elif str(raw_img_id) in self.features_db:
            h5_key = str(raw_img_id)
        else:
            raise KeyError(f"Image ID {raw_img_id} completely missing from H5!")

        # Read actual visual features
        visual_feats = torch.tensor(self.features_db[f'{h5_key}/features'][()])
        visual_boxes = torch.tensor(self.features_db[f'{h5_key}/boxes'][()])

        inputs = self.tokenizer(
            question_text,
            max_length=self.config.MAX_TEXT_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = self.tokenizer(
            answer_text,
            max_length=self.config.MAX_ANS_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids

        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "visual_feats": visual_feats,
            "visual_boxes": visual_boxes,
            "input_ids": inputs.input_ids.squeeze(),
            "attention_mask": inputs.attention_mask.squeeze(),
            "labels": labels.squeeze(),
            "raw_answer": answer_text
        }

print("Module 5: Dataset Class Defined")

In [ ]:
import os
import json
import h5py
import gc
from tqdm.auto import tqdm

LOCAL_DIR = "/content"
# %cd {LOCAL_DIR}

print("--- 1. Freeing Locked Files ---")
# Force garbage collection to release any Python locks on H5 files
gc.collect()
# Force delete the locked target file so h5py can create a fresh one safely
!rm -f resplit_val_obj36.h5

print("\n--- 2. Compacting with Correct ID Format ---")
with open('vqa_valid.json', 'r') as f:
    val_data = json.load(f)

required_ids = {int(item['img_id']) for item in val_data if 'img_id' in item}

# Safely open the source and create a fresh target file
with h5py.File('val2014_obj36.h5', 'r') as source_f, h5py.File('resplit_val_obj36.h5', 'w') as target_f:
    found_count = 0
    for img_id in tqdm(required_ids, desc="Compacting HDF5"):
        # The MAGIC FIX for Compaction: Adding the required UNC MSCOCO prefix
        search_key = f"COCO_val2014_{img_id:012d}"

        if search_key in source_f:
            grp = target_f.create_group(search_key)
            for feature_key in source_f[search_key].keys():
                grp.create_dataset(feature_key, data=source_f[f"{search_key}/{feature_key}"][()])
            found_count += 1

    print(f"✅ Compaction complete! Found {found_count} out of {len(required_ids)} images.")

print("\n--- 3. Final Cleanup ---")
# Delete the massive 8GB raw file to free up your Colab disk!
!rm -rf mscoco_imgfeat val2014_obj36.zip val2014_obj36.tsv val2014_obj36.h5
print("✅ ALL SET! The validation dataset is perfectly prepared. You can now run Module 6.")

## 6. The Ultimate Training & Evaluation Pipeline

In [ ]:
import random
from torch.utils.data import Subset

print("--- Initializing Tokenizer and DataLoaders ---")
tokenizer = T5Tokenizer.from_pretrained(cfg.MODEL_NAME)

LOCAL_ROOT = "/content"

TRAIN_H5 = f"{LOCAL_ROOT}/train2014_obj36.h5"
TRAIN_JSON = f"{LOCAL_ROOT}/vqa_train.json"
VAL_H5 = f"{LOCAL_ROOT}/resplit_val_obj36.h5"
VAL_JSON = f"{LOCAL_ROOT}/vqa_valid.json"

# --- Load and Sample Training Dataset ---
full_train_dataset = CustomVQADataset(TRAIN_H5, TRAIN_JSON, tokenizer, is_train=True)
TRAIN_SAMPLE_SIZE = 40000
train_indices = random.sample(range(len(full_train_dataset)), TRAIN_SAMPLE_SIZE)
train_dataset = Subset(full_train_dataset, train_indices)
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
print(f"✅ Subsampled Training Set: {len(train_dataset)} instances.")

# --- Load and Sample Validation Dataset ---
full_val_dataset = CustomVQADataset(VAL_H5, VAL_JSON, tokenizer, is_train=False)
VAL_SAMPLE_SIZE = 10000
val_indices = random.sample(range(len(full_val_dataset)), VAL_SAMPLE_SIZE)
val_dataset = Subset(full_val_dataset, val_indices)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
print(f"✅ Subsampled Validation Set: {len(val_dataset)} instances.")

# If you are worried about compute limits, you can remove "full" from this list
METHODS = ["lora", "dora", "full"]
experiment_results = {m: {"losses": [], "val_acc": 0.0, "params_m": 0.0} for m in METHODS}

for method in METHODS:
    print(f"\n{'='*50}\nSTARTING EXPERIMENT: {method.upper()}\n{'='*50}")

    # 1. Initialize Model
    model = CustomVLT5(cfg).to(cfg.DEVICE)

    # 2. Inject Fine-tuning Method
    target_modules = ["q", "k", "v", "o", "wi", "wo"]
    rep_cnt, trn_p, tot_p = inject_and_unfreeze(
        model.t5, target_modules, rank=cfg.RANK, alpha=cfg.ALPHA, dropout=cfg.DROPOUT, method=method
    )

    # IMPORTANT: Ensure visual layers are always trainable
    model.visual_embedding.requires_grad_(True)

    current_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    experiment_results[method]["params_m"] = current_trainable / 1e6
    print(f"[{method.upper()}] Trainable Params: {current_trainable/1e6:.2f}M")

    # 3. Training Loop Configuration

    param_optimizer = list(model.named_parameters())

    # We do not decay biases, LayerNorms, or the DoRA magnitude vector
    no_decay = ['bias', 'layer_norm', 'magnitude']

    optimizer_grouped_parameters = [
        {
            # Apply weight decay to standard multi-dimensional weights (like LoRA A and B)
            'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay) and p.requires_grad],
            'weight_decay': 0.01
        },
        {
            # NO weight decay for 1D parameters to preserve representation capacity
            'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay) and p.requires_grad],
            'weight_decay': 0.0
        }
    ]

    # Initialize AdamW with the grouped parameters
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=cfg.LR)

    model.train()

    step_losses = []
    LOG_EVERY_N_STEPS = 50  # Record loss every 50 batches for a smooth curve
    global_step = 0

    for epoch in range(cfg.EPOCHS):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}")
        for batch in pbar:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(cfg.DEVICE)
            attn_mask = batch["attention_mask"].to(cfg.DEVICE)
            vis_feats = batch["visual_feats"].to(cfg.DEVICE)
            vis_boxes = batch["visual_boxes"].to(cfg.DEVICE)
            labels = batch["labels"].to(cfg.DEVICE)

            outputs = model(input_ids, attn_mask, vis_feats, vis_boxes, labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            # Record step-level loss
            if global_step % LOG_EVERY_N_STEPS == 0:
                step_losses.append(loss.item())

            global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # Save fine-grained step losses instead of epoch losses
    experiment_results[method]["losses"] = step_losses

    # 4. Evaluation Loop
    print(f"Evaluating {method.upper()} on validation subset...")
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Eval [{method.upper()}]"):
            input_ids = batch["input_ids"].to(cfg.DEVICE)
            attn_mask = batch["attention_mask"].to(cfg.DEVICE)
            vis_feats = batch["visual_feats"].to(cfg.DEVICE)
            vis_boxes = batch["visual_boxes"].to(cfg.DEVICE)
            raw_answers = batch["raw_answer"]

            generated_ids = model.generate(input_ids, attn_mask, vis_feats, vis_boxes, max_length=cfg.MAX_ANS_LEN)
            preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            for p, t in zip(preds, raw_answers):
                if p.strip().lower() == t.strip().lower():
                    correct += 1
                total += 1

    acc = (correct / total) * 100 if total > 0 else 0
    experiment_results[method]["val_acc"] = acc
    print(f"[{method.upper()}] Final Accuracy: {acc:.2f}%")

    # Free memory before loading the next model
    del model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n✅ ALL EXPERIMENTS FINISHED.")


# Module 7: Visualization & Results Comparison

print("Generating evaluation charts...")

methods_list = list(experiment_results.keys())
params_list = [experiment_results[m]["params_m"] for m in methods_list]
acc_list = [experiment_results[m]["val_acc"] for m in methods_list]

colors = {"full": "#2ca02c", "lora": "#1f77b4", "dora": "#d62728"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Trainable Parameters
axes[0].bar(methods_list, params_list, color=[colors[m] for m in methods_list])
axes[0].set_ylabel("Trainable Parameters (M)")
axes[0].set_title("Trainable Parameter Count")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
for i, v in enumerate(params_list):
    axes[0].text(i, v, f"{v:.2f}M", ha='center', va='bottom', fontweight='bold')

# Plot 2: Training Loss Convergence (Using Step-level data)
for m in methods_list:
    losses = experiment_results[m]["losses"]
    if losses:
        # Create a smooth line and a transparent raw line
        smoothed = pd.Series(losses).ewm(alpha=0.1).mean()
        axes[1].plot(smoothed, label=m.upper(), color=colors[m], linewidth=2.5)
        axes[1].plot(losses, alpha=0.2, color=colors[m])

axes[1].set_title("Training Loss Convergence")
axes[1].set_xlabel("Steps (x50)")
axes[1].set_ylabel("Cross Entropy Loss")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

# Plot 3: Validation Accuracy
axes[2].bar(methods_list, acc_list, color=[colors[m] for m in methods_list])
axes[2].set_ylabel("VQA Accuracy (%)")
axes[2].set_title("Final Model Accuracy")
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
for i, v in enumerate(acc_list):
    axes[2].text(i, v, f"{v:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/vqa_finetuning_comparison.png")
plt.show()

print("✅ All Results Visualized and Saved to 'vqa_finetuning_comparison.png'.")

## 7. Visualization & Results Comparison

In [ ]:
methods_list = list(experiment_results.keys())
params_list = [experiment_results[m]["params_m"] for m in methods_list]
acc_list = [experiment_results[m]["val_acc"] for m in methods_list]

colors = {"full": "#2ca02c", "lora": "#1f77b4", "dora": "#d62728"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Trainable Parameters
axes[0].bar(methods_list, params_list, color=[colors[m] for m in methods_list])
axes[0].set_ylabel("Trainable Parameters (M)")
axes[0].set_title("Trainable Parameter Count")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
for i, v in enumerate(params_list):
    axes[0].text(i, v, f"{v:.2f}M", ha='center', va='bottom', fontweight='bold')

# Plot 2: Training Loss Convergence
for m in methods_list:
    losses = experiment_results[m]["losses"]
    if losses:
        # Create a smooth line and a transparent raw line
        smoothed = pd.Series(losses).ewm(alpha=0.3).mean()
        axes[1].plot(smoothed, label=m.upper(), color=colors[m], linewidth=2.5)
        axes[1].plot(losses, alpha=0.2, color=colors[m])

axes[1].set_title("Training Loss Convergence")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Cross Entropy Loss")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.7)

# Plot 3: Validation Accuracy
axes[2].bar(methods_list, acc_list, color=[colors[m] for m in methods_list])
axes[2].set_ylabel("VQA Accuracy (%)")
axes[2].set_title("Final Model Accuracy")
axes[2].grid(axis="y", linestyle="--", alpha=0.5)
for i, v in enumerate(acc_list):
    axes[2].text(i, v, f"{v:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("vqa_finetuning_comparison.png", dpi=300)
plt.show()

print("✅ All Results Visualized and Saved.")